In [1]:
from openai import OpenAI
from IPython.display import Markdown, display
from pypdf import PdfReader
import gradio as gr
from pydantic import BaseModel
from dotenv import load_dotenv
import uuid
import os

c:\Users\USER\Documents\NovaPay Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

True

In [3]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"]="1"
os.environ["OpenAI_API_KEY"]=os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"]="https://openrouter.ai/api/v1"


In [4]:
from agents import Agent, Runner, handoff, function_tool

In [5]:
def load_resources(folder_path: str) -> dict[str, str]:
    """Read all .pdf and .txt files in a folder, return {filename: text}."""
    documents = {}

    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)

        if filename.lower().endswith(".pdf"):
            reader = PdfReader(filepath)
            text = ""
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
            documents[filename] = text

        elif filename.lower().endswith(".txt"):
            with open(filepath, "r", encoding="utf-8") as f:
                documents[filename] = f.read()

    return documents


resources = load_resources("Resources")

### Enquiries and General Questions

In [6]:
@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents (PDF and TXT files) for relevant content matching the query."""
    query_lower = query.lower()
    matches = []

    for filename, text in resources.items():
        if query_lower in text.lower():
            # Find the matching snippet with some surrounding context
            idx = text.lower().find(query_lower)
            start = max(0, idx - 150)
            end = min(len(text), idx + 150)
            snippet = text[start:end].strip()
            matches.append(f"[{filename}]: ...{snippet}...")

    if not matches:
        return f"No matching content found for '{query}' in available documents."

    return "\n\n".join(matches)

@function_tool
def search_database(query: str) -> str:
    """ Search comapny database for customer info. """
    return f" Database results for '{query}': ..."


billing_agent = Agent(
    name="BillingSpecialist",
    instructions="""You handle billing questions:
    - Payment issues
    - Refunds
    - Invoice requests
    Be helpful and resolve issues quickly.""",
    tools=[search_documents]
)

technical_agent = Agent(
    name="TechnicalSupport",
    instructions="""You handle technical issues:
    - Bug reports
    - How-to questions
    - Feature requests
    Ask clarifying questions to understand the issue."""
)

General_agent=Agent(
    name="SupportSpecialist",
    instructions="""You are the first point of contact. Route customers immediately:
- Billing/payment/charge issues → transfer to BillingSpecialist RIGHT AWAY
- Technical problems → transfer to TechnicalSupport RIGHT AWAY

Do NOT ask clarifying questions if the intent is already clear. Transfer immediately.""",
    model="gpt-4o-mini",
    handoffs=[ 
        handoff(billing_agent),
        handoff(technical_agent),
]
)


### NovaPay_wallet 

In [7]:

@function_tool
def airtime(phone_number: str, network_provider: str, amount: int) -> str:
    return f" ₦{amount} {(network_provider).upper()}  to {phone_number}"

Airtime_agent = Agent(
name = "AirtimeAgent",
instructions="You help users buy airtime. "
        "Before calling the tool, always confirm: "
        "(1) the phone number, and "
        "(2) the amount. "
        "(3) the network provider"
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call airtime until both are confirmed. "
        "If the amount requested exceeds the available balance, tell the user INSUFFICIENT FUND."
    ,
tools=[search_database, airtime]
) 

@function_tool
def data_purchase(phone_number: str, network_provider: str, amount: int) -> str:
    return f" Your {network_provider.upper()} data bundle of ₦{amount} to {phone_number} is successfful "

Data_agent = Agent(
name = "DataAgent",
instructions=        "You help users buy data. "
        "Before calling the tool, always confirm: "
        "(1) the phone number, and "
        "(2) the amount. "
        "(3) the network provider"
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call data_purchase until both are confirmed. "
        "If the amount requested exceeds the available balance, tell the user INSUFFICIENT FUND.",
tools=[search_database, data_purchase]
) 

@function_tool
def money_transfer_intra(account_number: int, amount: int, bank: str) -> str:
    return f"₦{amount} sent to {account_number}"

transactionIntra = Agent(
name = "MoneyTransferIntra",
instructions="You help users send money to another NovaPay wallet (no charge). "
        "Before calling the tool, always confirm: "
        "(1) the amount, "
        "(2) the recipient's account number, "
        "(3) the recipient's bank name, and "
        "(4) that the account name matches the receiving end. "
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call money_transfer_intra until all four are confirmed. "
        "If the amount requested exceeds the available balance, tell the user INSUFFICIENT FUND.",
tools=[search_database, money_transfer_intra]
) 

@function_tool
def money_transfer_external(account_number: int, amount: int, bank: str) -> str:
    return f"₦{amount + 10} sent to {account_number}"

transactionExternal = Agent(
name = "MoneyTransferExternal",
instructions="You help users send money to other banks, with a ₦10 charge for NovaPay. "
        "Before calling the tool, always confirm: "
        "(1) the amount, "
        "(2) the recipient's account number, "
        "(3) the recipient's bank name, and "
        "(4) that the account name matches the receiving end. "
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call money_transfer_external until all four are confirmed by the user. "
        "If the amount requested exceeds the available balance, tell the user INSUFFICIENT FUND."
   ,
tools=[search_database, money_transfer_external]
) 

@function_tool
def bill_payment(bill_id: str, bill_type: str, top_up_amount: int) -> str:
    return f"Paid ₦{top_up_amount + (top_up_amount*0.1)} to {bill_id}"

bill_agent = Agent(
name = "billpayment",
instructions=        "You help users top up bill-based services (e.g. DSTV, GOTV, NEPA, WAEC/NECO tokens). "
        "Before calling the tool, always confirm: "
        "(1) the bill_id (which service/account is being topped up), and "
        "(2) the top_up_amount. "
        "(3) bill_type (getting to know if it is DSTV, GOTV or other related services)"
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call bill_payment until both are confirmed. "
        "If the amount requested exceeds the available balance, tell the user INSUFFICIENT FUND.",
tools=[search_database, bill_payment]
) 


NovaWalletAgent=Agent(
    name="WalletAgentSpecialist",
    instructions="""You are responsible for wallet transactions. Your man responsiblities are  for instant transfers, bill payments, airtime, and data purchases.
Route customers immediately:
- For Airtime recharge → transfer to AirtimeAgent RIGHT AWAY
- For Data Subscription → transfer to DataAgent RIGHT AWAY
- For money transfer to NovaPay wallet(Intra-Bank) → transfer to MoneyTransferIntra RIGHT AWAY
- For money transfer to bank or wallet(Other Bank) → transfer to MoneyTransferExternal RIGHT AWAY
- For bill payment, such as DSTV recharge, GOTV recharge, Nepa bill payment WAEC & NECO Toekn buying → transfer to billpayment RIGHT AWAY

Do NOT ask clarifying questions if the intent is already clear. Transfer immediately.""",
    model="gpt-4o-mini",
    handoffs=[ 
        handoff(Airtime_agent),
        handoff(Data_agent),
        handoff(transactionIntra),
        handoff(transactionExternal),
        handoff(bill_agent)
    ]
)


### NovaPay Business

In [8]:
@function_tool
def create_payment_link(amount: int, description: str, merchant_email: str) -> str:
    """Create a NovaPay payment link for a merchant. Always confirm before calling."""
    # TODO: Replace with real API call to NovaPay backend
    link_id = str(uuid.uuid4())[:8]
    return f"Payment link created: https://pay.novapay.ng/{link_id} for ₦{amount}. Desc: {description}"

payment_link_agent = Agent(
    name="PaymentLinkSpecialist",
    instructions="You handle NovaPay Payment Links. Use search_documents for fees/FAQ. "
        "Before calling the tool, always confirm: "
        "(1) the amount, "
        "(2) a description of what the payment is for, and "
        "(3) the merchant's email address. "
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call create_payment_link until all three are confirmed.",
    tools=[search_documents, create_payment_link]
)

@function_tool
def order_pos_terminal(merchant_name: str, address: str) -> str:
    """Order Android POS terminal. T+0 settlement."""
    return f"POS ordered for {merchant_name} to {address}. Ref: POS_{uuid.uuid4()[:6]}"

pos_agent = Agent(
    name="POSSpecialist",
    instructions=        "You handle NovaPay POS Terminals (Android devices, T+0 settlement). "
        "Use search_documents for specs. "
        "Before calling the tool, always confirm: "
        "(1) the merchant's name, and "
        "(2) the delivery address. "
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call order_pos_terminal until both are confirmed.",
    tools=[search_documents, order_pos_terminal]
)

checkout_agent = Agent(
name="CheckoutSpecialist",
instructions="You handle NovaPay Online Checkout for WooCommerce/Shopify. Use search_business_docs for setup guides.",
tools=[search_documents]
)

business_triage = Agent(
    name="BusinessTriage",
    instructions="""You route NovaPay Business requests. Do NOT solve.
- Payment links, pay-by-link, fees -> PaymentLinkSpecialist
- POS, terminal, hardware -> POSSpecialist
- Online store, WooCommerce, Shopify, embed -> CheckoutSpecialist
Transfer immediately. No questions.""",
    model="gpt-4o-mini",
    handoffs=[ 
        handoff(payment_link_agent),
        handoff(pos_agent), 
        handoff(checkout_agent)
    ]
)

#
#@function_tool
#def merchant_payment(tool_type: str, merchant_id: str, amount: int) -> str:
 #   return f"Processed {tool_type} payment of ₦{amount + (amount * 0.14)} for merchant {merchant_id}"

#NovaPayBusinessAgent = Agent(
#    name="NovaPayBusinessAgent",
#    instructions="You help merchants accept payments via payment links, POS terminals, and online checkout. Always confirm merchant_id, tool_type (link, POS, checkout), and amount before calling the tool. If the amount requested is not up to available balance, tell user INSUFFICIENT FUND.",
#    tools=[search_database, merchant_payment]
#)


### NovaPay API

In [9]:
@function_tool
def generate_api_key(company_name: str, email: str) -> str:
    """Generate NovaPay API test keys for a developer. Always confirm first."""
    return f"Test keys generated for {company_name}. Email sent to {email}. Docs: docs.novapay.ng"

api_agent = Agent(
    name="APISpecialist",
    instructions=(
         "You handle NovaPay API. NovaPay API lets businesses accept card and bank-transfer payments in apps/websites. "
        "Use search_documents for endpoints, fees, and docs. "
        "Before issuing keys, always confirm: "
        "(1) the company name, and "
        "(2) the email address. "
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call generate_api_key until both are confirmed."),
    tools=[search_documents, generate_api_key]
)


@function_tool
def api_payment(app_id: str, method: str, amount: int) -> str:
    return f"Processed {method} payment of ₦{amount} for app {app_id}"

NovaPayAPIAgent = Agent(
    name="NovaPayAPIAgent",
    instructions="You help developers accept card and bank-transfer payments inside their apps and websites. "
        "Before calling the tool, always confirm: "
        "(1) the app_id, "
        "(2) the payment method (card or bank-transfer), and "
        "(3) the amount. "
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call api_payment until all three are confirmed. "
        "If the amount requested exceeds the available balance, tell the user INSUFFICIENT FUND.",
    tools=[search_database, api_payment]
)

api_triage = Agent(
name="APITriage",
instructions="""You help developers pay for API and also help them receive money on their website. Do NOT solve.
- API, developer, integration, keys, endpoints -> APISpecialist
- Receiving money on their webesite -> NovaPayAPIAgent
Transfer immediately. No questions.""",
model="gpt-4o-mini",
handoffs=[ 
    handoff(api_agent), 
    handoff(NovaPayAPIAgent)
]
)


### NovaPay Save

In [10]:
@function_tool
def lock_funds(amount: int, duration_days: int, user_email: str) -> str:
    """Lock funds in NovaSave for a period to earn interest. Always confirm first."""
    plan_id = str(uuid.uuid4())[:8]
    return f"₦{amount} locked for {duration_days} days. Plan ID: NS_{plan_id}. Interest starts T+1."

novasave_agent = Agent(
name="NovaSaveSpecialist",
instructions=(
        "You handle NovaSave. NovaSave lets users lock funds and earn interest over a chosen period. "
        "Use search_documents for rates, min/max limits, and early withdrawal policy. "
        "Before calling the tool, always confirm: "
        "(1) the amount to lock, "
        "(2) the duration in days, and "
        "(3) the user's email address. "
        "If the user already provided any of these in their message, do not ask again for that item — "
        "only ask for whatever is missing. "
        "Do not call lock_funds until all three are confirmed."
),
tools=[search_documents, lock_funds]
)

save_triage = Agent(
name="SaveTriage",
instructions="""You route NovaPay product requests. Do NOT solve.
- Savings, lock funds, interest, fixed plan -> NovaSaveSpecialist
Transfer immediately. No questions.""",
model="gpt-4o-mini",
handoffs=[ 
    handoff(novasave_agent)
]
)

### Traige Agent

In [11]:
triage_agent = Agent(
name="MasterTriage",
instructions="""You are the first point of contact for NovaPay.
- Wallet: airtime, data, transfers, bills -> WalletTriage
- Business/Merchant: payment links, POS, checkout, onboarding -> BusinessTriage
- Saving: Locking up funds and earing interest -> SaveTriage
- API: Buying API and collecting money on website for API owners -> APISpecialist
- Billing/refunds/invoices -> BillingSpecialist
- Technical issues -> TechnicalSupport
Transfer immediately. No clarifying questions if intent is clear.""",
model="gpt-4o-mini",
handoffs=[  
handoff(NovaWalletAgent),
handoff(business_triage), 
handoff(api_triage),
handoff(save_triage),
handoff(billing_agent),
handoff(technical_agent),
]
)

result = await Runner.run(triage_agent, "I want to send 500000")
print(result.final_output)

I can help with that. Please provide the recipient’s:

- account number
- bank name
- account name (so I can confirm it matches)

You’ve already given the amount: ₦500,000.


In [13]:
import asyncio

def chat_fn(message, history):
       result = asyncio.run(Runner.run(triage_agent, message))
       return result.final_output

In [ ]:
demo = gr.ChatInterface(
    fn=chat_fn,
    title="NovaPay Assistant",
    description="Ask about wallet transfers, airtime, data, bills, business tools, API access, or savings.",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\USER\Documents\NovaPay Chatbot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\USER\Documents\NovaPay Chatbot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\USER\Documents\NovaPay Chatbot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\USER\Documents\NovaPay Chatbot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT'